# 06: Scoreline probabilities and evaluation

In [22]:
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss

df = pd.read_csv("../data/processed/model_table.csv",
                 parse_dates=["Date"], dtype={"Season": str})
df["h_rest"] = df["h_rest"].clip(upper=14)
df["a_rest"] = df["a_rest"].clip(upper=14)

features = [
    "h_points_last5", "h_gf5", "h_ga5", "h_gf10", "h_ga10", "h_rest",
    "a_points_last5", "a_gf5", "a_ga5", "a_gf10", "a_ga10", "a_rest",
]
train = df[df["Season"] <= "2324"]
valid = df[df["Season"] == "2425"]

def fit_poisson(target):
    model = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1.0, max_iter=1000))
    model.fit(train[features], train[target])
    return model

model_home = fit_poisson("home_goals")
model_away = fit_poisson("away_goals")
lam_home = model_home.predict(valid[features])
lam_away = model_away.predict(valid[features])

Scoreline grid

In [23]:
MAX_GOALS = 10
goals = np.arange(MAX_GOALS + 1)

def scoreline_grid(lam_h, lam_a):
    """Array of shape (matches, home goals, away goals) with scoreline probabilities."""
    p_h = poisson.pmf(goals[None, :], np.asarray(lam_h)[:, None])   # (n, 11)
    p_a = poisson.pmf(goals[None, :], np.asarray(lam_a)[:, None])   # (n, 11)
    return p_h[:, :, None] * p_a[:, None, :]                        # (n, 11, 11)

grid = scoreline_grid(lam_home, lam_away)
grid.shape, grid.sum(axis=(1, 2)).min()

((375, 11, 11), np.float64(0.999891162147293))

From scorelines to win/draw/loss

In [24]:
I, J = np.meshgrid(goals, goals, indexing="ij")    # I = home goals, J = away goals

def result_probs(lam_h, lam_a):
    g = scoreline_grid(lam_h, lam_a)
    t = g.sum(axis=(1, 2))
    return np.column_stack([(g * (I > J)).sum(axis=(1, 2)) / t,     # home win
                            (g * (I == J)).sum(axis=(1, 2)) / t,    # draw
                            (g * (I < J)).sum(axis=(1, 2)) / t])    # away win

probs = pd.DataFrame(result_probs(lam_home, lam_away),
                     columns=["H", "D", "A"], index=valid.index)
probs.sum(axis=1).round(6).unique()      # should be [1.]

array([1.])

In [25]:
k = 0
print(valid.iloc[k][["HomeTeam", "AwayTeam", "home_goals", "away_goals"]].to_dict())
print(f"expected goals: {lam_home[k]:.2f} - {lam_away[k]:.2f}")
print(probs.iloc[k].round(3).to_dict())
best = np.unravel_index(grid[k].argmax(), grid[k].shape)
print("most likely scoreline:", best, "with probability", round(grid[k][best], 3))

{'HomeTeam': 'Man United', 'AwayTeam': 'Fulham', 'home_goals': 1, 'away_goals': 0}
expected goals: 1.65 - 1.25
{'H': 0.467, 'D': 0.242, 'A': 0.29}
most likely scoreline: (np.int64(1), np.int64(1)) with probability 0.114


In [26]:
# Only matches with bookmaker odds, so every model is judged on the same rows
mask = valid[["B365H", "B365D", "B365A"]].notna().all(axis=1)
v = valid[mask]
y = v["FTR"].to_numpy()
labels = ["H", "D", "A"]

def metrics(name, p):
    p = np.asarray(p)
    pred = np.array(labels)[p.argmax(axis=1)]
    true_idx = pd.Series(y).map({"H": 0, "D": 1, "A": 2}).to_numpy()
    prob_of_truth = np.clip(p[np.arange(len(p)), true_idx], 1e-12, None)
    return {"model": name,
            "accuracy": (pred == y).mean(),
            "log loss": -np.log(prob_of_truth).mean()}

In [27]:
# 1. Constant: the training result shares for every match (its accuracy = "always home win")
shares = train["FTR"].value_counts(normalize=True)[labels]
p_const = np.tile(shares.to_numpy(), (len(v), 1))

# 2. Baseline goals: the same average goals for every match, pushed through the same grid
p_base = result_probs(np.full(len(v), train["home_goals"].mean()),
                      np.full(len(v), train["away_goals"].mean()))

# 3. Your model
p_model = result_probs(lam_home[mask.to_numpy()], lam_away[mask.to_numpy()])

# 4. Bookmaker: invert the odds, then remove the margin by normalising
raw = 1 / v[["B365H", "B365D", "B365A"]].to_numpy()
p_book = raw / raw.sum(axis=1, keepdims=True)

table = pd.DataFrame([
    metrics("Constant (training result shares)", p_const),
    metrics("Baseline goals -> Poisson", p_base),
    metrics("Poisson regression", p_model),
    metrics("Bookmaker odds (Bet365)", p_book),
]).round(3)
table

,model,accuracy,log loss
0,Constant (training result shares),0.411,1.078
1,Baseline goals -> Poisson,0.411,1.078
2,Poisson regression,0.496,1.019
3,Bookmaker odds (Bet365),0.541,0.971


In [28]:
best_idx = grid.reshape(len(grid), -1).argmax(axis=1)
pred_h, pred_a = np.divmod(best_idx, MAX_GOALS + 1)
exact = ((pred_h == valid["home_goals"].to_numpy()) &
         (pred_a == valid["away_goals"].to_numpy())).mean()
print(f"Exact scoreline hit rate: {exact:.1%}")
pd.Series(list(zip(pred_h, pred_a))).value_counts().head()

Exact scoreline hit rate: 12.3%


(1, 1)    305
(2, 0)     31
(1, 0)     19
(2, 1)     10
(0, 2)      7
Name: count, dtype: int64

In [29]:
calib = pd.DataFrame({
    "predicted share": probs.mean(),
    "actual share": valid["FTR"].value_counts(normalize=True)[labels],
}).round(3)
calib

,predicted share,actual share
H,0.442,0.411
D,0.236,0.240
A,0.322,0.349


In [35]:
import numpy as np
import pandas as pd
from scipy.stats import poisson

def check_match(home, away, season="2425", my_guess=None, max_goals=10):
    """Predict a match that has already been played and compare it with the real result."""
    rows = df[(df["Season"] == season) & (df["HomeTeam"] == home) & (df["AwayTeam"] == away)]
    if rows.empty:
        teams = sorted(df.loc[df["Season"] == season, "HomeTeam"].unique())
        raise ValueError(f"No {home} vs {away} match found in {season}. Team names: {teams}")
    r = rows.iloc[0]

    def outcome(h, a):
        return "H" if h > a else "A" if h < a else "D"

    # 1. Expected goals from the two Poisson models (features use only earlier matches)
    X = rows[features].iloc[[0]]
    lam_h = model_home.predict(X)[0]
    lam_a = model_away.predict(X)[0]

    # 2. Scoreline grid: rows = home goals, columns = away goals
    goals = np.arange(max_goals + 1)
    grid = np.outer(poisson.pmf(goals, lam_h), poisson.pmf(goals, lam_a))
    grid = grid / grid.sum()

    # 3. Result probabilities
    I, J = np.indices(grid.shape)
    masks = {"H": I > J, "D": I == J, "A": I < J}
    probs = {k: grid[m].sum() for k, m in masks.items()}
    names = {"H": f"{home} win", "D": "Draw", "A": f"{away} win"}
    predicted = max(probs, key=probs.get)

    # 4. Predicted score: most likely scoreline within the predicted result
    sub = np.where(masks[predicted], grid, 0)
    ph, pa = np.unravel_index(sub.argmax(), grid.shape)

    # 5. Bookmaker probabilities with the margin removed (if odds exist)
    odds = r[["B365H", "B365D", "B365A"]].to_numpy(dtype=float)
    book = None
    if not np.isnan(odds).any():
        raw = 1 / odds
        book = dict(zip(["H", "D", "A"], raw / raw.sum()))

    # 6. Actual result from the data
    hg, ag, actual = int(r["home_goals"]), int(r["away_goals"]), r["FTR"]

    # 7. Report
    top = np.argsort(grid.ravel())[::-1][:5]
    top_scores = [divmod(k, max_goals + 1) for k in top]

    print(f"{home} vs {away}  |  {r['Date'].date()}  |  season {season}")
    print("-" * 64)
    print(f"Expected goals:  {home} {lam_h:.2f}  -  {lam_a:.2f} {away}")
    print("Model:      " + " | ".join(f"{names[k]} {probs[k]:.1%}" for k in "HDA"))
    if book:
        print("Bookmaker:  " + " | ".join(f"{names[k]} {book[k]:.1%}" for k in "HDA"))
    print("Most likely scorelines: " + ", ".join(f"{i}-{j} ({grid[i, j]:.1%})" for i, j in top_scores))
    print("-" * 64)
    print(f"MODEL'S PREDICTED SCORE:  {home} {ph}-{pa} {away}  ({names[predicted]}, scoreline probability {grid[ph, pa]:.1%})")
    if my_guess is not None:
        gh, ga = my_guess
        print(f"YOUR GUESS:               {home} {gh}-{ga} {away}  ({names[outcome(gh, ga)]}, model probability {grid[gh, ga]:.1%})")
    print("-" * 64)
    print(f"ACTUAL:  {home} {hg}-{ag} {away}  ->  {names[actual]}")

    def verdict(label, h, a):
        exact = "yes" if (h, a) == (hg, ag) else "no"
        res = "yes" if outcome(h, a) == actual else "no"
        err = abs(h - hg) + abs(a - ag)
        print(f"{label:<6} exact score: {exact:<3} | correct result: {res:<3} | total goal error: {err}")

    verdict("Model", ph, pa)
    if my_guess is not None:
        verdict("You", gh, ga)
    print(f"Probability the model gave to the actual result: {probs[actual]:.1%}")
    if hg <= max_goals and ag <= max_goals:
        print(f"Probability the model gave to the exact score {hg}-{ag}: {grid[hg, ag]:.1%}")

check_match("Chelsea", "Arsenal", my_guess=(1, 3))

Chelsea vs Arsenal  |  2024-11-10  |  season 2425
----------------------------------------------------------------
Expected goals:  Chelsea 1.66  -  1.25 Arsenal
Model:      Chelsea win 47.0% | Draw 24.2% | Arsenal win 28.8%
Bookmaker:  Chelsea win 35.1% | Draw 27.8% | Arsenal win 37.1%
Most likely scorelines: 1-1 (11.3%), 2-1 (9.4%), 1-0 (9.1%), 2-0 (7.5%), 1-2 (7.1%)
----------------------------------------------------------------
MODEL'S PREDICTED SCORE:  Chelsea 2-1 Arsenal  (Chelsea win, scoreline probability 9.4%)
YOUR GUESS:               Chelsea 1-3 Arsenal  (Arsenal win, model probability 2.9%)
----------------------------------------------------------------
ACTUAL:  Chelsea 1-1 Arsenal  ->  Draw
Model  exact score: no  | correct result: no  | total goal error: 1
You    exact score: no  | correct result: no  | total goal error: 2
Probability the model gave to the actual result: 24.2%
Probability the model gave to the exact score 1-1: 11.3%
